# 加载并使用保存好的垃圾短信分类模型

这个 notebook 演示如何在一个新的 session 中加载第 6 章训练并保存的 `spam_classifier.pth`，然后直接对短信文本做 `spam` / `not spam` 分类。

> 运行前请先确认原 notebook 已经执行过保存模型的代码：`torch.save(model.state_dict(), "spam_classifier.pth")`。

In [1]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken

# 与训练 notebook 保持一致
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12,
}

MAX_LENGTH = 120  # 第 6 章训练集中的最长短信 token 数
PAD_TOKEN_ID = 50256

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
print(f"使用设备: {device}")

使用设备: cpu


In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1),
        )

    def forward(self, x):
        b, num_tokens, _ = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask[:num_tokens, :num_tokens].bool()
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = F.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vecs = attn_weights @ values
        context_vecs = context_vecs.transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vecs)


class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi, device=x.device)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["drop_rate"],
            num_heads=cfg["n_heads"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

In [3]:
def find_model_path():
    candidates = [
        Path("spam_classifier.pth"),
        Path("ch06_classification_finetuning") / "spam_classifier.pth",
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        "找不到 spam_classifier.pth。请先在原 notebook 中运行保存模型的单元，"
        "或把模型文件放到当前 notebook 同一目录。"
    )


model = GPTModel(BASE_CONFIG)
model.out_head = nn.Linear(BASE_CONFIG["emb_dim"], 2)  # 2 类：not spam / spam

model_path = find_model_path()
try:
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
except TypeError:
    state_dict = torch.load(model_path, map_location=device)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

print(f"模型已从 {model_path} 加载完成")

模型已从 spam_classifier.pth 加载完成


In [4]:
def classify_text(text, model, tokenizer, device, max_length=MAX_LENGTH, pad_token_id=PAD_TOKEN_ID):
    """返回预测标签、置信度和两个类别的概率。"""
    model.eval()

    input_ids = tokenizer.encode(text)
    supported_context_length = model.pos_emb.weight.shape[0]
    max_length = min(max_length, supported_context_length)

    input_ids = input_ids[:max_length]
    input_ids += [pad_token_id] * (max_length - len(input_ids))
    input_tensor = torch.tensor(input_ids, device=device).unsqueeze(0)

    with torch.no_grad():
        logits = model(input_tensor)[:, -1, :]
        probabilities = torch.softmax(logits, dim=-1).squeeze(0)
        predicted_label = torch.argmax(probabilities).item()

    label_name = "spam" if predicted_label == 1 else "not spam"
    confidence = probabilities[predicted_label].item()
    return {
        "label": label_name,
        "confidence": confidence,
        "not_spam_probability": probabilities[0].item(),
        "spam_probability": probabilities[1].item(),
    }

In [5]:
examples = [
    "You are a winner! Claim your free prize now by calling this number.",
    "Hey, are we still meeting for dinner tonight?",
]

for text in examples:
    result = classify_text(text, model, tokenizer, device)
    print(f"文本: {text}")
    print(f"预测: {result['label']} | 置信度: {result['confidence']:.2%}")
    print(
        f"not spam: {result['not_spam_probability']:.2%}, "
        f"spam: {result['spam_probability']:.2%}"
    )
    print("-" * 60)

文本: You are a winner! Claim your free prize now by calling this number.
预测: spam | 置信度: 78.61%
not spam: 21.39%, spam: 78.61%
------------------------------------------------------------
文本: Hey, are we still meeting for dinner tonight?
预测: not spam | 置信度: 99.63%
not spam: 99.63%, spam: 0.37%
------------------------------------------------------------


## 使用自己的短信

修改下面的 `my_text`，然后重新运行单元即可。

In [6]:
my_text = "恭喜你! 你中大奖了，请访问 weixin.qq.com 领取你的奖品"

classify_text(my_text, model, tokenizer, device)

{'label': 'spam',
 'confidence': 0.7122482061386108,
 'not_spam_probability': 0.28775179386138916,
 'spam_probability': 0.7122482061386108}

## 中文短信测试

下面构造一些中文垃圾短信和正常短信，看看这个只用英文 SMS 数据微调过的模型能不能泛化到中文。注意：模型没有用中文垃圾短信训练过，所以这里的结果只能作为实验观察，不能代表真实中文场景的可靠效果。

In [7]:
chinese_examples = [
    {
        "text": "恭喜您获得1000元现金红包，请立即点击链接领取，逾期作废。",
        "expected": "spam",
    },
    {
        "text": "尊敬的用户，您的账户积分即将清零，回复88领取免费提现额度。",
        "expected": "spam",
    },
    {
        "text": "限时福利！免费领取VIP会员资格，点击 http://example.com 立即激活。",
        "expected": "spam",
    },
    {
        "text": "您好，您有一笔快递异常，请登录链接填写银行卡信息完成验证。",
        "expected": "spam",
    },
    {
        "text": "今晚七点一起吃饭吗？我已经订好餐厅了。",
        "expected": "not spam",
    },
    {
        "text": "妈妈，我下班后去超市买菜，你还需要什么吗？",
        "expected": "not spam",
    },
    {
        "text": "会议改到下午三点，地点还是原来的会议室。",
        "expected": "not spam",
    },
    {
        "text": "你的外卖已经到楼下了，方便的话现在下来取一下。",
        "expected": "not spam",
    },
]

correct = 0
for item in chinese_examples:
    result = classify_text(item["text"], model, tokenizer, device)
    is_correct = result["label"] == item["expected"]
    correct += int(is_correct)

    print(f"文本: {item['text']}")
    print(f"预期: {item['expected']} | 模型预测: {result['label']} | {'✓' if is_correct else '✗'}")
    print(
        f"not spam: {result['not_spam_probability']:.2%}, "
        f"spam: {result['spam_probability']:.2%}"
    )
    print("-" * 80)

print(f"中文样例准确率: {correct}/{len(chinese_examples)} = {correct / len(chinese_examples):.2%}")

文本: 恭喜您获得1000元现金红包，请立即点击链接领取，逾期作废。
预期: spam | 模型预测: spam | ✓
not spam: 12.83%, spam: 87.17%
--------------------------------------------------------------------------------


文本: 尊敬的用户，您的账户积分即将清零，回复88领取免费提现额度。
预期: spam | 模型预测: spam | ✓
not spam: 12.00%, spam: 88.00%
--------------------------------------------------------------------------------
文本: 限时福利！免费领取VIP会员资格，点击 http://example.com 立即激活。
预期: spam | 模型预测: spam | ✓
not spam: 15.52%, spam: 84.48%
--------------------------------------------------------------------------------
文本: 您好，您有一笔快递异常，请登录链接填写银行卡信息完成验证。
预期: spam | 模型预测: spam | ✓
not spam: 20.93%, spam: 79.07%
--------------------------------------------------------------------------------


文本: 今晚七点一起吃饭吗？我已经订好餐厅了。
预期: not spam | 模型预测: spam | ✗
not spam: 39.23%, spam: 60.77%
--------------------------------------------------------------------------------


文本: 妈妈，我下班后去超市买菜，你还需要什么吗？
预期: not spam | 模型预测: not spam | ✓
not spam: 50.08%, spam: 49.92%
--------------------------------------------------------------------------------
文本: 会议改到下午三点，地点还是原来的会议室。
预期: not spam | 模型预测: spam | ✗
not spam: 46.86%, spam: 53.14%
--------------------------------------------------------------------------------
文本: 你的外卖已经到楼下了，方便的话现在下来取一下。
预期: not spam | 模型预测: not spam | ✓
not spam: 51.48%, spam: 48.52%
--------------------------------------------------------------------------------
中文样例准确率: 6/8 = 75.00%


## 更多中文压力测试

这一组样例覆盖更多中文场景：中奖诈骗、金融/贷款营销、快递/账户异常、链接诱导，以及正常生活和工作短信。

In [8]:
more_chinese_examples = [
    # 垃圾短信：中奖/红包/免费领取
    {"text": "恭喜您被抽中为幸运用户，点击链接领取888元现金红包。", "expected": "spam", "type": "中奖红包"},
    {"text": "您有一份新年礼包待领取，回复 Y 立即获取兑换码。", "expected": "spam", "type": "中奖红包"},
    {"text": "免费领取苹果手机一台，名额有限，先到先得。", "expected": "spam", "type": "中奖红包"},
    {"text": "系统检测到您获得专属福利，打开短链接即可提现。", "expected": "spam", "type": "中奖红包"},

    # 垃圾短信：金融/贷款/账户异常
    {"text": "无需抵押，极速放款，最高可借20万元，点击申请。", "expected": "spam", "type": "金融贷款"},
    {"text": "您的信用额度已提升至50000元，请登录页面完成激活。", "expected": "spam", "type": "金融贷款"},
    {"text": "银行卡账户存在风险，请立即输入验证码解除冻结。", "expected": "spam", "type": "账户异常"},
    {"text": "您的网银证书即将失效，请点击链接更新身份信息。", "expected": "spam", "type": "账户异常"},

    # 垃圾短信：快递/服务异常
    {"text": "您的快递因地址异常无法派送，请点击链接重新填写收货信息。", "expected": "spam", "type": "快递异常"},
    {"text": "您的包裹海关清关失败，请支付手续费后继续配送。", "expected": "spam", "type": "快递异常"},
    {"text": "尊敬的会员，您的积分今晚清零，立即兑换免费提现券。", "expected": "spam", "type": "营销诱导"},
    {"text": "限时秒杀开始，回复 1 获取内部优惠链接。", "expected": "spam", "type": "营销诱导"},

    # 正常短信：生活场景
    {"text": "我已经到地铁站了，大概十分钟后到你家。", "expected": "not spam", "type": "正常生活"},
    {"text": "今天有点堵车，晚饭可能要晚一点开始。", "expected": "not spam", "type": "正常生活"},
    {"text": "记得带伞，天气预报说下午会下雨。", "expected": "not spam", "type": "正常生活"},
    {"text": "冰箱里还有鸡蛋和牛奶，不用再买了。", "expected": "not spam", "type": "正常生活"},

    # 正常短信：工作/学习场景
    {"text": "明天上午十点开项目同步会，麻烦提前看一下文档。", "expected": "not spam", "type": "正常工作"},
    {"text": "代码我已经提交了，你有空帮忙 review 一下。", "expected": "not spam", "type": "正常工作"},
    {"text": "老师说作业截止时间延后到周五晚上。", "expected": "not spam", "type": "正常学习"},
    {"text": "这篇论文我看完了，晚上可以一起讨论实验部分。", "expected": "not spam", "type": "正常学习"},
]

results = []
for item in more_chinese_examples:
    prediction = classify_text(item["text"], model, tokenizer, device)
    is_correct = prediction["label"] == item["expected"]
    results.append({**item, **prediction, "correct": is_correct})

correct = sum(row["correct"] for row in results)
print(f"更多中文样例准确率: {correct}/{len(results)} = {correct / len(results):.2%}\n")

for row in results:
    mark = "✓" if row["correct"] else "✗"
    print(f"[{row['type']}] {row['text']}")
    print(f"预期: {row['expected']} | 预测: {row['label']} | {mark}")
    print(f"not spam: {row['not_spam_probability']:.2%}, spam: {row['spam_probability']:.2%}")
    print("-" * 80)

更多中文样例准确率: 15/20 = 75.00%

[中奖红包] 恭喜您被抽中为幸运用户，点击链接领取888元现金红包。
预期: spam | 预测: spam | ✓
not spam: 9.32%, spam: 90.68%
--------------------------------------------------------------------------------
[中奖红包] 您有一份新年礼包待领取，回复 Y 立即获取兑换码。
预期: spam | 预测: spam | ✓
not spam: 29.11%, spam: 70.89%
--------------------------------------------------------------------------------
[中奖红包] 免费领取苹果手机一台，名额有限，先到先得。
预期: spam | 预测: spam | ✓
not spam: 36.39%, spam: 63.61%
--------------------------------------------------------------------------------
[中奖红包] 系统检测到您获得专属福利，打开短链接即可提现。
预期: spam | 预测: spam | ✓
not spam: 24.80%, spam: 75.20%
--------------------------------------------------------------------------------
[金融贷款] 无需抵押，极速放款，最高可借20万元，点击申请。
预期: spam | 预测: spam | ✓
not spam: 11.85%, spam: 88.15%
--------------------------------------------------------------------------------
[金融贷款] 您的信用额度已提升至50000元，请登录页面完成激活。
预期: spam | 预测: spam | ✓
not spam: 12.51%, spam: 87.49%
--------------------------------------------